# Лабораторная работа №3: K-ближайшие соседи (KNN)
## Датасет: Mushroom (съедобные и ядовитые грибы)
**Выполнил:** Хрипков Тимофей, группа ИУ5-65Б


**Цель работы:** изучение способов подготовки выборки и подбора гиперпараметров на примере метода ближайших соседей.

**Датасет:** Mushroom (OpenML ID: 24) содержит 8124 образцов грибов с 22 категориальными признаками. Задача бинарной классификации: определить, съедобен гриб (edible) или ядовит (poisonous).

**План работы:**
1. Загрузка и анализ данных
2. Предобработка (кодирование категориальных признаков)
3. Разделение на обучающую и тестовую выборки
4. Масштабирование признаков
5. Обучение базовой модели KNN (K=5)
6. Подбор гиперпараметров с помощью GridSearchCV и RandomizedSearchCV
7. Сравнение результатов базовой и оптимальной моделей

In [ ]:
# Ячейка 1: Импорт библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

## 1. Загрузка и первичный анализ данных

Загружаем датасет Mushroom через OpenML. Датасет не содержит пропусков, все признаки являются категориальными.

In [ ]:
# Ячейка 2: Загрузка и первичный анализ датасета mushroom
data = fetch_openml(name='mushroom', version=1, as_frame=True, parser='auto')
df = data.frame

print("=== ИНФОРМАЦИЯ О ДАТАСЕТЕ ===\n")
print(f"Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов")
print(f"\nЦелевая переменная 'class':")
print(df['class'].value_counts())
print(f"\nПропуски в данных: {df.isnull().sum().sum()}")

df.info()

=== ИНФОРМАЦИЯ О ДАТАСЕТЕ ===

Размер датасета: 8124 строк, 23 столбцов

Целевая переменная 'class':
class
e    4208
p    3916
Name: count, dtype: int64

Пропуски в данных: 2480
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8124 entries, 0 to 8123
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   cap-shape                 8124 non-null   category
 1   cap-surface               8124 non-null   category
 2   cap-color                 8124 non-null   category
 3   bruises%3F                8124 non-null   category
 4   odor                      8124 non-null   category
 5   gill-attachment           8124 non-null   category
 6   gill-spacing              8124 non-null   category
 7   gill-size                 8124 non-null   category
 8   gill-color                8124 non-null   category
 9   stalk-shape               8124 non-null   category
 10  stalk-root                5644 non-nul

## 2. Предобработка данных

Поскольку KNN основан на вычислении расстояний между объектами, необходимо:
1. Закодировать категориальные признаки с помощью One-Hot Encoding
2. Закодировать целевую переменную (edible → 1, poisonous → 0)

In [ ]:
# Ячейка 3: Предобработка данных
# В датасете mushroom все признаки категориальные, пропусков нет
X = df.drop('class', axis=1)
y = df['class']

# One-Hot кодирование категориальных признаков
X_encoded = pd.get_dummies(X, drop_first=True)

# Кодирование целевой переменной: edible (съедобный) -> 1, poisonous (ядовитый) -> 0
y_encoded = y.map({'e': 1, 'p': 0})  # e = edible, p = poisonous

print(f"Исходное количество признаков: {X.shape[1]}")
print(f"Количество признаков после One-Hot кодирования: {X_encoded.shape[1]}")
print(f"\nРаспределение целевой переменной после кодирования:\n{y_encoded.value_counts()}")
print("  1 - съедобный, 0 - ядовитый")

# Покажем несколько строк закодированных данных
display(X_encoded.head())

Исходное количество признаков: 22
Количество признаков после One-Hot кодирования: 94

Распределение целевой переменной после кодирования:
class
1    4208
0    3916
Name: count, dtype: int64
  1 - съедобный, 0 - ядовитый


,cap-shape_c,cap-shape_f,cap-shape_k,cap-shape_s,cap-shape_x,cap-surface_g,cap-surface_s,cap-surface_y,cap-color_c,cap-color_e,...,population_n,population_s,population_v,population_y,habitat_g,habitat_l,habitat_m,habitat_p,habitat_u,habitat_w
0,False,False,False,False,True,False,True,False,False,False,...,False,True,False,False,False,False,False,False,True,False
1,False,False,False,False,True,False,True,False,False,False,...,True,False,False,False,True,False,False,False,False,False
2,False,False,False,False,False,False,True,False,False,False,...,True,False,False,False,False,False,True,False,False,False
3,False,False,False,False,True,False,False,True,False,False,...,False,True,False,False,False,False,False,False,True,False
4,False,False,False,False,True,False,True,False,False,False,...,False,False,False,False,True,False,False,False,False,False


## 3. Разделение выборки и масштабирование

Разделяем данные на обучающую (80%) и тестовую (20%) выборки с сохранением пропорций классов (stratify). Затем применяем StandardScaler для нормализации признаков — это важно, так как KNN использует расстояния между точками.

In [ ]:
# Ячейка 4: Разделение на обучающую и тестовую выборки + масштабирование
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Масштабирование (важно для KNN, так как используются расстояния)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Размер обучающей выборки: {X_train_scaled.shape}")
print(f"Размер тестовой выборки: {X_test_scaled.shape}")
print(f"\nРаспределение классов в обучающей выборке:")
print(f"  Съедобные (1): {(y_train == 1).sum()}")
print(f"  Ядовитые (0): {(y_train == 0).sum()}")

Размер обучающей выборки: (6499, 94)
Размер тестовой выборки: (1625, 94)

Распределение классов в обучающей выборке:
  Съедобные (1): 3366
  Ядовитые (0): 3133


## 4. Базовая модель KNN (K=5)

Обучим модель K-ближайших соседей с произвольно выбранным гиперпараметром K=5. Оценим качество с помощью accuracy и отчета о классификации.

In [ ]:
# Ячейка 5: Базовая модель KNN с K=5
clf_base = KNeighborsClassifier(n_neighbors=5)
clf_base.fit(X_train_scaled, y_train)

y_pred_base = clf_base.predict(X_test_scaled)

print("=== БАЗОВАЯ МОДЕЛЬ (K = 5) ===\n")
print(f"Accuracy на тестовой выборке: {accuracy_score(y_test, y_pred_base):.4f}\n")
print("Подробный отчет о классификации:")
print(classification_report(y_test, y_pred_base, target_names=['Ядовитый (0)', 'Съедобный (1)']))

=== БАЗОВАЯ МОДЕЛЬ (K = 5) ===

Accuracy на тестовой выборке: 0.9994

Подробный отчет о классификации:
               precision    recall  f1-score   support

 Ядовитый (0)       1.00      1.00      1.00       783
Съедобный (1)       1.00      1.00      1.00       842

     accuracy                           1.00      1625
    macro avg       1.00      1.00      1.00      1625
 weighted avg       1.00      1.00      1.00      1625



## 5. Подбор гиперпараметров с помощью кросс-валидации

Для поиска оптимальных гиперпараметров используем две стратегии:

1. **StratifiedKFold + GridSearchCV** — полный перебор всех комбинаций параметров с 5-кратной стратифицированной кросс-валидацией
2. **StratifiedShuffleSplit + RandomizedSearchCV** — случайный поиск с 5-кратным стратифицированным случайным разбиением

**Перебираемые гиперпараметры:**
- `n_neighbors`: нечетные числа от 1 до 49
- `weights`: 'uniform' (все соседи равнозначны) или 'distance' (соседи взвешиваются по расстоянию)
- `metric`: 'euclidean' (евклидово расстояние) или 'manhattan' (манхэттенское расстояние)

In [ ]:
# Ячейка 6: Подбор гиперпараметров с помощью кросс-валидации
# Уменьшенная сетка для ускорения (на 8k строк это всё равно займет несколько минут)
param_grid = {
    'n_neighbors': np.arange(1, 51, 2),  # 1, 3, 5, ..., 49 (25 значений)
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

print("=== СТРАТЕГИЯ 1: StratifiedKFold + GridSearchCV ===")
print(f"Всего комбинаций для перебора: {len(param_grid['n_neighbors']) * 2 * 2}")
print("Это займет примерно 2-3 минуты...\n")

cv_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=cv_stratified,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_scaled, y_train)

print("\n=== СТРАТЕГИЯ 2: StratifiedShuffleSplit + RandomizedSearchCV ===")
cv_shuffle = StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
random_search = RandomizedSearchCV(
    KNeighborsClassifier(),
    param_distributions=param_grid,
    n_iter=20,  # 20 случайных комбинаций
    cv=cv_shuffle,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)
random_search.fit(X_train_scaled, y_train)

print("\n=== РЕЗУЛЬТАТЫ ПОДБОРА ===\n")
print(f"Лучшие параметры (GridSearchCV): {grid_search.best_params_}")
print(f"Accuracy на кросс-валидации (GridSearchCV): {grid_search.best_score_:.4f}\n")

print(f"Лучшие параметры (RandomizedSearchCV): {random_search.best_params_}")
print(f"Accuracy на кросс-валидации (RandomSearch): {random_search.best_score_:.4f}")

=== СТРАТЕГИЯ 1: StratifiedKFold + GridSearchCV ===
Всего комбинаций для перебора: 100
Это займет примерно 2-3 минуты...

Fitting 5 folds for each of 100 candidates, totalling 500 fits

=== СТРАТЕГИЯ 2: StratifiedShuffleSplit + RandomizedSearchCV ===
Fitting 5 folds for each of 20 candidates, totalling 100 fits

=== РЕЗУЛЬТАТЫ ПОДБОРА ===

Лучшие параметры (GridSearchCV): {'metric': 'euclidean', 'n_neighbors': np.int64(1), 'weights': 'uniform'}
Accuracy на кросс-валидации (GridSearchCV): 1.0000

Лучшие параметры (RandomizedSearchCV): {'weights': 'distance', 'n_neighbors': np.int64(3), 'metric': 'manhattan'}
Accuracy на кросс-валидации (RandomSearch): 1.0000


## 6. Сравнение базовой и оптимальной моделей

Оценим лучшую модель, найденную с помощью GridSearchCV, на отложенной тестовой выборке и сравним её с базовой моделью (K=5).

In [ ]:
# Ячейка 7: Сравнение базовой и оптимальной модели на тестовой выборке
best_knn = grid_search.best_estimator_
y_pred_best = best_knn.predict(X_test_scaled)

print("=== СРАВНЕНИЕ МОДЕЛЕЙ НА ТЕСТОВОЙ ВЫБОРКЕ ===\n")
print(f"Базовая модель (K=5):")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_base):.4f}\n")

print(f"Оптимальная модель {grid_search.best_params_}:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_best):.4f}\n")

print("ДЕТАЛЬНЫЙ ОТЧЕТ ДЛЯ ОПТИМАЛЬНОЙ МОДЕЛИ:")
print(classification_report(y_test, y_pred_best, target_names=['Ядовитый (0)', 'Съедобный (1)']))

=== СРАВНЕНИЕ МОДЕЛЕЙ НА ТЕСТОВОЙ ВЫБОРКЕ ===

Базовая модель (K=5):
  Accuracy: 0.9994

Оптимальная модель {'metric': 'euclidean', 'n_neighbors': np.int64(1), 'weights': 'uniform'}:
  Accuracy: 1.0000

ДЕТАЛЬНЫЙ ОТЧЕТ ДЛЯ ОПТИМАЛЬНОЙ МОДЕЛИ:
               precision    recall  f1-score   support

 Ядовитый (0)       1.00      1.00      1.00       783
Съедобный (1)       1.00      1.00      1.00       842

     accuracy                           1.00      1625
    macro avg       1.00      1.00      1.00      1625
 weighted avg       1.00      1.00      1.00      1625

